# 큐싱 탐지 — URLBERT 4종 분류 헤드 학습·평가 (자급식 Kaggle 노트북)

코드가 이 노트북 안에 들어 있고, PhiUSIIL·OpenPhish·Tranco는 노트북이 직접 내려받습니다.
**필요한 것:** Accelerator GPU, Internet On, Add Input → `sid321axn/malicious-urls-dataset`

결과물: `/kaggle/working/qsing_artifacts.zip`(서버용), `/kaggle/working/qsing_reports.zip`(지표·리포트)

In [ ]:
# 0. 내장 코드 풀기 + 패키지 설치
import base64, io, os, sys, zipfile, subprocess, shutil
CODE = '/kaggle/working/qsing_code'
shutil.rmtree(CODE, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64 := '...'))).extractall(CODE)
sys.path.insert(0, CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tldextract', 'tranco'], check=False)
import transformers, torch
from packaging.version import Version
if Version(transformers.__version__) < Version('4.48.0'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.48'], check=True)
    raise SystemExit('transformers 업그레이드 완료 — Restart & Run All 하세요')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)

In [ ]:
# 0-1. 단위 테스트
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', os.path.join(CODE, 'tests')], capture_output=True, text=True, cwd=CODE)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, '단위 테스트 실패'

## 외부 데이터 다운로드 (PhiUSIIL, OpenPhish, Tranco) — URL에는 접속하지 않고 목록 파일만 받습니다

In [ ]:
import datetime, json, urllib.request
INP = '/kaggle/working/inputs'
os.makedirs(INP, exist_ok=True)
today = datetime.date.today().isoformat()
meta = {'collected_at': today}

def fetch(urls, out):
    for u in urls:
        try:
            req = urllib.request.Request(u, headers={'User-Agent': 'qsing-research-notebook'})
            with urllib.request.urlopen(req, timeout=120) as r, open(out, 'wb') as f:
                shutil.copyfileobj(r, f)
            if os.path.getsize(out) > 0:
                return u
        except Exception as e:
            print('  실패:', u, type(e).__name__, e)
    return None

# PhiUSIIL (UCI)
z = os.path.join(INP, 'phiusiil.zip')
src = fetch(['https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip'], z)
if src:
    with zipfile.ZipFile(z) as zf:
        zf.extractall(os.path.join(INP, 'phiusiil'))
    meta['phiusiil'] = {'source': src}
print('PhiUSIIL:', meta.get('phiusiil'))

# OpenPhish Community Feed (비상업적 이용)
op = os.path.join(INP, f'openphish_{today.replace("-", "")}.txt')
src = fetch(['https://raw.githubusercontent.com/openphish/public_feed/refs/heads/main/feed.txt',
             'https://raw.githubusercontent.com/openphish/public_feed/main/feed.txt',
             'https://openphish.com/feed.txt'], op)
if src:
    meta['openphish'] = {'source': src, 'lines': sum(1 for _ in open(op, encoding='utf-8', errors='replace'))}
else:
    os.path.exists(op) and os.remove(op)
print('OpenPhish:', meta.get('openphish'))

# Tranco Top 1M (고정 list ID 기록)
tr = os.path.join(INP, 'tranco.csv')
try:
    from tranco import Tranco
    lst = Tranco(cache=True, cache_dir=os.path.join(INP, '.tranco')).list()
    with open(tr, 'w') as f:
        for i, d in enumerate(lst.top(100000), 1):
            f.write(f'{i},{d}\n')
    meta['tranco'] = {'list_id': lst.list_id, 'list_page': lst.list_page}
except Exception as e:
    print('  tranco 패키지 실패:', e)
    zt = os.path.join(INP, 'tranco.zip')
    if fetch(['https://tranco-list.eu/top-1m.csv.zip'], zt):
        with zipfile.ZipFile(zt) as zf:
            with zf.open('top-1m.csv') as s, open(tr, 'wb') as f:
                shutil.copyfileobj(s, f)
        meta['tranco'] = {'list_id': 'latest (top-1m.csv.zip)'}
print('Tranco:', meta.get('tranco'))
print('URLhaus: Auth-Key가 필요해 이번 실행에서는 건너뜀')
json.dump(meta, open(os.path.join(INP, 'collection_meta.json'), 'w'), ensure_ascii=False, indent=2)

## 설정 — 입력 파일 자동 탐색

In [ ]:
from qsing_model import pipeline as P
from qsing_model import data as D
ROOTS = ('/kaggle/input', INP)
cfg = P.Config(
    out_dir='/kaggle/working/qsing_out',
    collected_at=today,
    malicious_urls_csv=D.find_file(['malicious_phish*.csv', '*malicious*url*.csv'], roots=('/kaggle/input',)),
    phiusiil_csv=D.find_file(['phiusiil*.csv', '*phiusiil*.csv'], roots=ROOTS, required=False),
    urlhaus_csv=D.find_file(['*urlhaus*.csv'], roots=ROOTS, required=False),
    openphish_txts=D.find_files(['*openphish*.txt'], roots=ROOTS),
    tranco_csv=D.find_file(['tranco.csv', '*tranco*.csv', 'top-1m.csv'], roots=ROOTS, required=False),
)
for k in ('malicious_urls_csv', 'phiusiil_csv', 'urlhaus_csv', 'openphish_txts', 'tranco_csv'):
    print(f'{k:20s}', getattr(cfg, k))

## P0 환경 확인 + 원격 코드 검토 (저장소 커밋 고정)

In [ ]:
rep0 = P.p0_environment(cfg, show_code=True)

## 리허설 — 6천 건 표본으로 P1~P7 전체 점검

In [ ]:
s = P.smoke_config(cfg)
s.revision = cfg.revision
P.p1_prepare(s); P.p2_split(s); P.p3_baseline(s); P.p4_embed(s); P.p4_train(s); P.p6_thresholds(s); P.p5_evaluate(s)
P.p7_export(s, zip_path='/kaggle/working/_smoke_artifacts')
print('\n리허설 통과')

## P1 데이터 점검·전처리

In [ ]:
rep1 = P.p1_prepare(cfg)

## P2 도메인 그룹 분리

In [ ]:
rep2 = P.p2_split(cfg)

## P3 기준선

In [ ]:
base = P.p3_baseline(cfg)

## P4 임베딩 + 헤드 학습

In [ ]:
rep4 = P.p4_embed(cfg)

In [ ]:
sel = P.p4_train(cfg)

## P6 위험도 임계값

In [ ]:
thr = P.p6_thresholds(cfg)

## P5 평가

In [ ]:
metrics = P.p5_evaluate(cfg)

## P7 서버용 산출물

In [ ]:
exp = P.p7_export(cfg)

## 요약

In [ ]:
summary = P.run_summary_md(cfg)
print(summary)
shutil.copy(os.path.join(INP, 'collection_meta.json'), cfg.path('reports', 'collection_meta.json'))
shutil.make_archive('/kaggle/working/qsing_reports', 'zip', os.path.join(cfg.out_dir, 'reports'))
shutil.rmtree('/kaggle/working/qsing_out/emb', ignore_errors=True)   # 큰 임베딩 파일은 출력에서 제외
shutil.rmtree('/kaggle/working/qsing_out/_smoke', ignore_errors=True)
print('완료: /kaggle/working/qsing_artifacts.zip, /kaggle/working/qsing_reports.zip')